---
# Commande pour le rendu : quarto render main.ipynb --execute
# Instructions pour quarto
title: "Traitement des données IDFM"
format:
  html:
    code-fold: true
    embed-resources: true
---

Retour à la [page principale](main.html).

# Introduction
Introduction 

presentation des données

liens vers le PDF de presentation

But = pour chaque arret savoir le nombre d'arrets

# Scrapping

On recupère un zip et on ouvre avec pandas les tables pertinentes :

In [ ]:
# Téléchargement et ouverture des données GTFS IDFM

# Ensure the repository root is on sys.path so local modules can be imported
import sys
from pathlib import Path
project_root = Path.cwd().parent  # notebook is in notebooks/, repo root is its parent
print(f"Project root: {project_root}")
if str(project_root) not in sys.path:
	sys.path.insert(0, str(project_root))

# Import the helper using an absolute import (not a relative one)
from script.download_data import get_IDFM_data_path
import pandas as pd

file_names = get_IDFM_data_path()
print(file_names.keys())

# open only usefull df :
usefull_keys = ["routes", "trips", "stop_times", "stops", "calendar"]

idfm = {k: pd.read_csv(f) for k, f in file_names.items() if k in usefull_keys}

# Traitement

## Regroupement des arrêts
Première étape, pour chaque arret de transports en commun, les différents quais et arret de bus son précisément géolocalisés. On n'a pas besoin de cette granularité, on regroupe pour tous les arrets partageant le même nom en un seul arret (dit arret commercial).

In [ ]:
# on remplace stop_id par parent_station pour les arrêts parents dans stop_times et stops
# le but est d'obtenir une ligne par arret commercial (initialement: la gare de bus et la gare RER d'un même arret partagent le même nom d'arret commercial mais ont des stop_id différents)

print(f"Nombre initial de stop_id uniques : {idfm['stops']['stop_id'].nunique()} dans stops, {idfm['stop_times']['stop_id'].nunique()} dans stop_times")

# les arrets parents n'ont pas de parent_station, on remplace les NA par leur propre stop_id
nb_na_before = idfm["stops"]["parent_station"].isna().sum()
idfm["stops"].fillna({"parent_station": idfm["stops"]["stop_id"]}, inplace=True)

# on remplace stop_id par parent_station dans stop_times
idfm["stop_times"] = idfm["stop_times"].merge(
    idfm["stops"][["stop_id", "parent_station"]], on="stop_id", how="left"
).drop(columns=["stop_id"]).rename(columns={"parent_station": "stop_id"})

# on ne conserve que les arrêts parents dans stops
idfm["stops"] = idfm["stops"][idfm["stops"]["parent_station"] == idfm["stops"]["stop_id"]].reset_index(drop=True)


print(f"Nombre final de stop_id uniques : {idfm['stops']['stop_id'].nunique()} dans les deux tables")

## Selection des trajets d'un jour donné
On souhaite un nombre de passages par jour, pour cela on prend un lundi présent dans les données et on ne conserve que les trajets qui sont effectués ce jour là

In [ ]:
# On ne conserve que les services ayant un jour donné pour calculer un nombre de trajet sur une journée en semaine

# on prend le prochain lundi à partir d'aujourd'hui au format YYYYMMDD
from datetime import datetime, timedelta
today = datetime.today()
next_monday = today + timedelta(days=(7 - today.weekday()) % 7)
next_monday_str = next_monday.strftime("%Y%m%d")
print(f"Travail avec le jour: {next_monday_str}")

next_monday_str = "20251215"  # reproductibilité # TODO

services_one_day = idfm["calendar"][(idfm["calendar"]["monday"] == 1) & (idfm["calendar"]["start_date"] <= int(next_monday_str)) & (idfm["calendar"]["end_date"] >= int(next_monday_str)) ]["service_id"]
trips_one_day = idfm["trips"].merge(services_one_day, on="service_id", how="inner")
print(f"{len(trips_one_day)} trajets  conservés, {len(idfm['trips']) - len(trips_one_day) } supprimés")

## Nombre de passage par arrêt
On calcul le nombre de passage de chaque ligne à chaque arrêt, un exemple de résultat et affiché pour cité universitaire

In [ ]:
# Pour chaque stop_id, et chaque route_id, on calcul le nombre d'arrets par jour à ce stop
# voir le pdf docu_gtfs.pdf, section 3.4.5 page 17 pour la structure des tables
stop_times_one_day = idfm["stop_times"].merge(trips_one_day[["trip_id", "route_id"]], on="trip_id", how="inner")
stop_route_counts = stop_times_one_day.groupby(["stop_id", "route_id"]).size().reset_index(name="nb_stops_per_day")
print(f"Nombre de paires (stop_id, route_id) uniques le sur le jour {next_monday_str} : {len(stop_route_counts)}")

# on ajoute les infos des arrêts
stop_route_counts = (
    	stop_route_counts.merge(
				idfm["stops"][["stop_id", "stop_name", "stop_lat", "stop_lon", "zone_id"]], 
				on="stop_id", how="left"
            )
        	.merge(
                idfm["routes"][["route_id", "route_short_name", "route_type"]],
                on="route_id", how="left"
            )
)


# Affiche les passages dans une gare donnée
stop_route_counts[stop_route_counts["stop_name"] == "Cité Universitaire"].sort_values(by="nb_stops_per_day", ascending=False)

## Traitement des erreurs sur les lignes de bus
TODO

## Regroupement par mode de transport


On crée une ligne par arret, qui synthétise les nombre de passage par heure pour chaque type de transport

Note : Les codes 6 et 7 (funiculaires et téléphériques) ne sont pas considérés

In [ ]:
# calcul du nombre de passage par type de transport
passage_par_arret = (stop_route_counts.groupby(["stop_id", "stop_name", "stop_lat", "stop_lon", "route_type"]) 
                    ["nb_stops_per_day"].sum().reset_index())
passage_par_arret = (passage_par_arret[passage_par_arret["route_type"].isin([0, 1, 2, 3])].reset_index(drop=True)
                    .pivot_table(index=["stop_id", "stop_name", "stop_lat", "stop_lon"], columns="route_type", values="nb_stops_per_day", fill_value=0))
# rename columns
passage_par_arret = passage_par_arret.rename(columns={
    0: "nb_tramway_per_day",
    1: "nb_metro_per_day",
    2: "nb_train_per_day",
    3: "nb_bus_per_day"
})

passage_par_arret = passage_par_arret.reset_index()
passage_par_arret.head(5)

# Enregistrement

On enregistre les résultats dans 

In [ ]:
# Enregistre passage_par_arret et stop_route_counts dans un fichier GeoJSON
import geopandas as gpd
from shapely.geometry import Point

# creation du dossier de sauvegarde s'il n'existe pas
dir_path = Path("cache/results")
dir_path.mkdir(parents=True, exist_ok=True)

# sauvegarde des deux geodataframes
geometry = [Point(xy) for xy in zip(passage_par_arret['stop_lon'], passage_par_arret['stop_lat'])]
gdf_synthetique = gpd.GeoDataFrame(passage_par_arret, geometry=gpd.points_from_xy(passage_par_arret["stop_lon"], passage_par_arret["stop_lat"]), crs="EPSG:4326")
gdf_synthetique.to_file(dir_path / "passage_par_arret_synthetique.geojson", driver="GeoJSON")

geometry = [Point(xy) for xy in zip(stop_route_counts['stop_lon'], stop_route_counts['stop_lat'])]
gdf_full = gpd.GeoDataFrame(stop_route_counts, geometry=geometry, crs="EPSG:4326")
gdf_full.to_file(dir_path / "passage_par_arret_full.geojson", driver="GeoJSON")

# Visualisation

On affiche sur une carte les arrêts du RER B, en utilisant folium

Creation du fond de carte :

In [ ]:
# Creation d'un fond de carte avec les limites des departements et communes d'IDF
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
from script.download_fond_carte import load_fonds_carte

coms, deps = load_fonds_carte(crs=4326, force_download=False)

# Listes filtrées de la "petite couronne"
PETITE_COURONNE = [75, 92, 93, 94]
coms_pc = coms[coms["INSEE_DEP"].astype(int).isin(PETITE_COURONNE)].copy()
deps_pc = deps[deps["INSEE_DEP"].astype(int).isin(PETITE_COURONNE)].copy()


# trace ces departements et communes avec folium
m = folium.Map(location=[48.84, 2.35], zoom_start=11, tiles="cartodb positron")

for _, row in deps_pc.iterrows():
    folium.GeoJson(
        row['geometry'],
        name=f"Departement {row['INSEE_DEP']}",
        style_function=lambda x: {
            'fillColor': 'none',
            'color': 'black',
            'weight': 2,
            'dashArray': '5, 5'
        }
    ).add_to(m)
for _, row in coms_pc.iterrows():
    folium.GeoJson(
        row['geometry'],
        name=f"Commune {row['INSEE_COM']}",
        style_function=lambda x: {
            'fillColor': 'none',
            'color': 'grey',
            'weight': 1,
            'dashArray': '2, 2'
        }
    ).add_to(m)

Affichage des arrets du RER B

In [ ]:
from numpy import log

# filtrer les gares du RER B
gares_a_afficher = gdf_synthetique.merge(
    gdf_full[(gdf_full['route_short_name'] == 'B') & (gdf_full['route_type'] == 2)][['stop_id']],
    on='stop_id',
)

# visualisation avec folium
import branca.colormap as cm

cmap = cm.LinearColormap(colors=['blue', 'red'], vmin=log(gares_a_afficher['nb_train_per_day']).min(), vmax=log(gares_a_afficher['nb_train_per_day']).max())

for _, row in gares_a_afficher.iterrows():
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=5,
        tooltip=f"{row['stop_name']}: {row['nb_train_per_day']} trains/jour",
        color=cmap(log(row['nb_train_per_day'])),
        fill=True,
        fill_color=cmap(log(row['nb_train_per_day']))
    ).add_to(m)

m
